# China Opportunity Context Atlas: a reproducible research walkthrough

**Weijia Xian · AI-assisted research software · v0.4**

Question: what can district data tell us when linked childhood-to-adulthood records are unavailable?

We reproduce two bounded experiments. Population sensitivity is not an income model. The Hong Kong prediction target is current household income, not intergenerational mobility or a causal effect. This notebook uses only Python's standard library and committed inputs.


In [1]:
from pathlib import Path
import sys, json, math
ROOT = Path.cwd() if (Path.cwd() / "dist/data/atlas-data.json").exists() else Path.cwd().parent
assert (ROOT / "scripts/run_experiments.py").exists(), "Start this notebook from the repository or notebooks directory"
sys.path.insert(0, str(ROOT / "scripts"))
from run_experiments import build, load_income, benchmark, sensitivity
atlas = json.loads((ROOT / "dist/data/atlas-data.json").read_text(encoding="utf-8"))
print({c["name"]: len(c["districts"]) for c in atlas["cities"]})
print("Units:", sum(len(c["districts"]) for c in atlas["cities"]))


{'北京': 16, '上海': 16, '广州': 11, '深圳': 9, '成都': 20, '香港': 18}
Units: 90


## 1. What is actually observed?

Inspect input fields and missingness before fitting anything. Hong Kong income and mainland population are intentionally separate layers. Several mainland units combine development zones, following source definitions.


In [2]:
for city in atlas["cities"]:
    print(city["name"], {key: sum(d.get(key) is not None for d in city["districts"])
                        for key in ("populationGrowthPct", "workingAgeShare", "childShare", "medianHouseholdIncome")})


北京 {'populationGrowthPct': 16, 'workingAgeShare': 16, 'childShare': 16, 'medianHouseholdIncome': 0}
上海 {'populationGrowthPct': 16, 'workingAgeShare': 15, 'childShare': 15, 'medianHouseholdIncome': 0}
广州 {'populationGrowthPct': 11, 'workingAgeShare': 11, 'childShare': 11, 'medianHouseholdIncome': 0}
深圳 {'populationGrowthPct': 9, 'workingAgeShare': 9, 'childShare': 9, 'medianHouseholdIncome': 0}
成都 {'populationGrowthPct': 20, 'workingAgeShare': 17, 'childShare': 17, 'medianHouseholdIncome': 0}
香港 {'populationGrowthPct': 0, 'workingAgeShare': 0, 'childShare': 0, 'medianHouseholdIncome': 18}


## 2. Weight sensitivity

Midrank percentiles are calculated within each city. We enumerate every nonnegative weight combination on a 0.1 simplex grid. There are 66 combinations. A district must have all three inputs; no imputation is used. Best/worst ranks are ranges over model assumptions, not confidence intervals.


In [3]:
city = next(c for c in atlas["cities"] if c["id"] == "chengdu")
result = sensitivity(city)
print("Scenarios:", result["scenarioCount"], "Complete units:", result["completeCount"])
for district in city["districts"]:
    row = result["districts"].get(district["id"])
    print(district["name"], "missing" if row is None else f"rank {row['bestRank']}–{row['worstRank']}; top-quarter scenario share {row['topQuarterShare']:.1%}")


Scenarios: 66 Complete units: 17
锦江区 rank 4–7; top-quarter scenario share 72.7%
青羊区 rank 4–8; top-quarter scenario share 47.0%
金牛区 rank 4–17; top-quarter scenario share 7.6%
武侯—高新片区 missing
成华区 rank 3–15; top-quarter scenario share 45.5%
龙泉驿区 rank 1–6; top-quarter scenario share 92.4%
青白江区 rank 6–9; top-quarter scenario share 0.0%
新都区 rank 1–5; top-quarter scenario share 100.0%
温江区 rank 1–4; top-quarter scenario share 100.0%
双流—天府片区 missing
郫都—高新片区 missing
新津区 rank 7–15; top-quarter scenario share 0.0%
金堂县 rank 3–14; top-quarter scenario share 18.2%
大邑县 rank 12–16; top-quarter scenario share 0.0%
蒲江县 rank 8–13; top-quarter scenario share 0.0%
都江堰市 rank 10–15; top-quarter scenario share 0.0%
彭州市 rank 10–17; top-quarter scenario share 0.0%
邛崃市 rank 12–17; top-quarter scenario share 0.0%
崇州市 rank 9–17; top-quarter scenario share 0.0%
简阳—东部新区 rank 1–17; top-quarter scenario share 16.7%


## 3. Temporal holdout on observed household income

Fit 2022–2023; select shrinkage weight on 2024 MAE; freeze the weight; recompute trends with 2022–2024; evaluate on 2025. No 2025 labels enter tuning or prediction.

The forecast is `last_income * exp(lambda * district_log_growth + (1-lambda) * mean_district_log_growth)`. The district mean growth is not growth of the all-Hong-Kong income median. Four fixed model families are compared. This is a retrospective exercise, not preregistration or a real-time forecast.


In [4]:
panel, names = load_income(ROOT / "data/source/hk-income-130-06806-api.json")
backtest = benchmark(panel, names)
print("Selected weight:", backtest["selectedWeight"])
print(f"{'Model':<14} {'MAE':>12} {'RMSE':>12} {'Bias':>12}")
for m in backtest["models"]:
    print(f"{m['id']:<14} {m['mae']:12.2f} {m['rmse']:12.2f} {m['bias']:12.2f}")


Selected weight: 0.4
Model                   MAE         RMSE         Bias
persistence          738.89      1052.25      -605.56
pooled               649.04       833.45        -7.70
local                878.79      1165.55       -49.61
shrinkage            721.87       956.02       -25.15


## 4. Challenge the leakage boundary

Perturb all held-out labels. The selected weight and predictions must stay identical while measured error changes.


In [5]:
import copy
changed = copy.deepcopy(panel)
for series in changed.values():
    series[2025] *= 4
perturbed = benchmark(changed, names)
assert backtest["selectedWeight"] == perturbed["selectedWeight"]
assert [d["predictions"] for d in backtest["districts"]] == [d["predictions"] for d in perturbed["districts"]]
assert backtest["models"][0]["mae"] != perturbed["models"][0]["mae"]
print("Passed: held-out labels cannot change predictions or tuning.")


Passed: held-out labels cannot change predictions or tuning.


## 5. Interpretation and next experiment

The pooled trend outperformed tuned shrinkage and independent trends on the 2025 holdout. This is a useful negative result: adding district-specific flexibility did not yield the lowest error. Eighteen correlated districts and one held-out year do not establish significance or future reliability. Official survey estimates also have sampling error that is not modeled here.

An actual opportunity atlas needs authorized parent–child links, childhood geography, consistently defined adult outcomes, survey-aware estimation and external validation. A separate credible design is required for causal effects. The existing population indicators do not solve that identification problem.

**Try modifying:** use RMSE on the validation set; document the change before examining test performance. Then ask whether the conclusion depends on the choice of loss. Reusing this test year repeatedly would turn it into a development set.


In [6]:
output = build()
print("Reproduced:", output["version"])
for path, digest in output["inputs"].items():
    print(path, digest)


Reproduced: 0.4.0
dist/data/atlas-data.json cb9480bac9254e27bd46dc93fe4e7847148bb4588b7c37f0fdec8175ac74d16d
data/source/hk-income-130-06806-api.json c357deec0b0d3346415955255bc961e5d95c758ba37b09f2205623842b483212
